In [10]:
import boto3
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

# Create a boto3 session in us-east-2
boto_session = boto3.Session(region_name="us-east-2")
sagemaker_session = sagemaker.Session(boto_session=boto_session)

role = "arn:aws:iam::222634404112:role/SageMakerExecutionRole"
job_name = "occupancy-model-t1"


[04/09/25 12:30:51] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=759908;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=335097;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/credentials.py#1352\1352]8;;\

In [11]:
sklearn_estimator = SKLearn(
    entry_point="train.py",       # Use your training script
    source_dir=".",               # Package the current directory (which includes train.py, inference.py, requirements.txt, etc.)
    dependencies=["requirements.txt"],
    role=role,
    instance_type="ml.m5.xlarge",
    framework_version="0.23-1",
    output_path="s3://dana-minicapstone/model-artifacts/",
    sagemaker_session=sagemaker_session,
    job_name=job_name
)

sklearn_estimator.fit()


                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=983322;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=103977;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[04/09/25 12:30:52] INFO     Creating training-job with name:                                       ]8;id=78143;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=751016;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#1042\1042]8;;\
                             sagemaker-scikit-learn-2025-04-09-19-30-51-799                                        

2025-04-09 19:30:53 Starting - Starting the training job...
2025-04-09 19:31:22 Starting - Preparing the instances for training...
2025-04-09 19:31:54 Downloading - Downloading the training image...
2025-04-09 19:32:30 Training - Training image download completed. Training in progress....
2025-04-09 19:33:00 Uploading - Uploading generated training model...
2025-04-09 19:33:13 Completed - Training job completed
..Training seconds: 94
Billable seconds: 94


In [12]:
from sagemaker.sklearn.model import SKLearnModel

# Use the model_data from your training job
model = SKLearnModel(
    model_data=sklearn_estimator.model_data,
    role=role,
    entry_point="inference.py",  # Reference your inference script
    framework_version="0.23-1",
    py_version="py3",
    source_dir="."  # Include any additional files if needed
)

predictor = model.deploy(initial_instance_count=1, instance_type="ml.m5.large")


[04/09/25 12:33:49] INFO     Creating model with name:                                              ]8;id=8392;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=471380;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#4094\4094]8;;\
                             sagemaker-scikit-learn-2025-04-09-19-33-49-415                                        

[04/09/25 12:33:50] INFO     Creating endpoint-config with name                                     ]8;id=796075;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=421140;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#6019\6019]8;;\
                             sagemaker-scikit-learn-2025-04-09-19-33-50-306                                        

                    INFO     Creating endpoint with name                                            ]8;id=683476;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=379225;file:///Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/session.py#4841\4841]8;;\
                             sagemaker-scikit-learn-2025-04-09-19-33-50-306                                        

------!

In [13]:
# Retrieve test CSV from S3 and then call the endpoint
s3 = boto3.client('s3', region_name='us-east-2')
bucket_name = "dana-minicapstone"
test_key = "data/hvac_test.csv"
response = s3.get_object(Bucket=bucket_name, Key=test_key)
test_csv = response['Body'].read().decode('utf-8')

# Call your endpoint with the CSV string
result = predictor.predict(test_csv)
print("Predictions from endpoint:\n", result)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6 test_csv = response['Body'].read().decode('utf-8')                                          │
│    7                                                                                             │
│    8 # Call your endpoint with the CSV string                                                    │
│ ❱  9 result = predictor.predict(test_csv)                                                        │
│   10 print("Predictions from endpoint:\n", result)                                               │
│   11                                                                                             │
│                                                                                                  │
│ /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sagemaker/base_p │
│ redictor.py:212 in predict                                                                       │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/client. │
│ py:570 in _api_call                                                                              │
│                                                                                                  │
│    567 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    568 │   │   │   │   )                                                                         │
│    569 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  570 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    571 │   │                                                                                     │
│    572 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    573                                                                                           │
│                                                                                                  │
│ /Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/botocore/context │
│ .py:124 in wrapper                                                                               │
│                                                                                                  │
│   121 │   │   │   with start_as_current_context():                                               │
│   122 │   │   │   │   if hook:                                                                   │
│   123 │   │   │   │   │   hook()                                                                 │
│ ❱ 124 │   │   │   │   return func(*args, **kwargs)                                               │
│   125 │   │                                                